In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

### 1. Define Schema

In [14]:
spark = SparkSession.builder \
    .appName("Sentiment140_Fixed") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

schema = StructType([
    StructField("target", IntegerType(), False),
    StructField("ids", LongType(), False),
    StructField("date", StringType(), False),
    StructField("flag", StringType(), False),
    StructField("user", StringType(), False),
    StructField("text", StringType(), False)
])

### 2. Load and Process Data

In [15]:
def load_and_process(path: str, split_name: str):
    df = spark.read.option("header", "true").option("inferSchema", "false").schema(schema).csv(path)

    # handling timezone suffix
    df = df.withColumn(
        "timestamp",
        to_timestamp(
            regexp_replace(col("date"), r" [A-Z]{3}$", ""),
            "EEE MMM dd HH:mm:ss yyyy"
        )
    ).withColumn("date_parsed", col("timestamp").cast("date"))

    # 0 to 0 (negative), 4 to 1 (positive); drop neutral (2) if binary
    df = df.filter(col("target").isin([0, 4])) \
           .withColumn("label", when(col("target") == 4, 1).otherwise(0)) \
           .drop("target")

    # anonymize + lightweight clean
    df = df.withColumn(
        "text_clean",
        trim(
            regexp_replace(
                regexp_replace(
                    regexp_replace(col("text"), r"http\S+", ""),     # URLs → ""
                    r"@\w+", "@USER"                                 # mentions → @USER
                ),
                r"\s+", " "                                          # collapse whitespace
            )
        )
    ).drop("text", "date").withColumnRenamed("text_clean", "text")

    # adding metadata
    return df \
        .withColumn("split", lit(split_name)) \
        .withColumn("version", lit("v1_header_clean")) \
        .withColumn("processed_utc", current_timestamp()) \
        .select(
            "ids", "label", "user", "text",
            "timestamp", "date_parsed", "flag", "split", "version", "processed_utc"
        )

In [ ]:
train1 = load_and_process("../data/raw_data/train_part1.csv", "train1")
train2 = load_and_process("../data/raw_data/train_part2.csv", "train2")
test   = load_and_process("../data/raw_data/test_data.csv",   "test")
val    = load_and_process("../data/raw_data/val_data.csv", "validation")

In [17]:
# let's combine our trains
train = train1.unionByName(train2).withColumn("split", lit("train"))

In [18]:
datasets = {
    "train": train,
    "test": test,
    "validation": val
}

In [19]:
# Show first 5 rows
train1.show(5, truncate=60)

+----------+-----+-------------+------------------------------------------------------------+---------+-----------+--------+------+---------------+-------------------------+
|       ids|label|         user|                                                        text|timestamp|date_parsed|    flag| split|        version|            processed_utc|
+----------+-----+-------------+------------------------------------------------------------+---------+-----------+--------+------+---------------+-------------------------+
|2051457557|    1|JGoldsborough|@USER WIth ya. &quot;I'd like a Palm Pre, Touchstone char...|     NULL|       NULL|NO_QUERY|train1|v1_header_clean|2025-11-14 10:58:53.76617|
|2053083567|    1|       Psioui|felt the earthquake this afternoon, it seems to be a , bu...|     NULL|       NULL|NO_QUERY|train1|v1_header_clean|2025-11-14 10:58:53.76617|
|1976779404|    1|    adriville|                  Ruffles on shirts are like so in, me Likey|     NULL|       NULL|NO_QUERY|train1